[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_english_hebrew_only.ipynb)


**1. Imports.** Install the public model package if needed and import all libraries.


In [ ]:
# 1. All imports and the single optional dependency installation.
import hashlib
import importlib.util
import io
import re
import subprocess
import sys
from urllib.request import urlopen

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer


**2. Model.** Load the pretrained LaBSE encoder on CPU.


In [ ]:
# 2. Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")


**3. Data.** Load and verify the full survey and metadata; display all 200 records.


In [ ]:
# 3. Download the complete survey and tagged metadata from a fixed GitHub commit.
# This is the HTTPS + SHA-256 method used in your older notebook; no SSH key is needed.
SOURCE_ROOT = (
    "https://raw.githubusercontent.com/shulruferor-sudo/"
    "Advance-Data-Science/81ae343dd70fb4165a64fd498f5ea0b2c6c86a73/"
    "Datasets/cbs_occupation_survey_sample/"
)

def load_csv(filename, sha256):
    with urlopen(SOURCE_ROOT + filename, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == sha256, filename
    return pd.read_csv(io.BytesIO(content), dtype="string", keep_default_na=False)

survey_df = load_csv(
    "survey_200_records_current.csv",
    "ea2c420880df19eb5a42ef8d4f3a4e3fd7349db57c65b1ab5d53d16e26d88dbd",
)
metadata_df = load_csv(
    "survey_metadata_he_en_names.csv",
    "2ad8b579bd1733960a7ed0c4552be0a908a2f31cadac3dd394e61f9937ae5ec6",
)

# Select columns by their metadata label N, not by a hardcoded list.
name_columns = metadata_df.loc[
    metadata_df["HE_EN_NAMES"].eq("N"), "משתנה"
].tolist()

assert survey_df.shape == (200, 18)
assert survey_df["ID"].is_unique
assert len(name_columns) == 7
assert set(name_columns).issubset(survey_df.columns)

print("Survey:", survey_df.shape, "| English/Hebrew fields:", name_columns)
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(survey_df)


**4. Inference.** In columns marked `N`, propose only the highest-scoring existing Hebrew name when cosine > α; keep the original data unchanged.


In [ ]:
# 4. Inference only: the best existing Hebrew label per English expression.
ALPHA = 0.50

# Detect English words, acronyms, names, and phrases inside any cell.
LATIN = re.compile(
    r"[A-Za-z]+(?:[&./'-][A-Za-z]+)*(?:\s+[A-Za-z]+(?:[&./'-][A-Za-z]+)*)*"
)
HEBREW = re.compile(r"[\u0590-\u05FF]")

# Preserve the survey. Only the copy receives proposed substitutions.
proposed_df = survey_df.copy()
scores = []
changed_cells = []

for column in name_columns:
    values = survey_df[column].fillna("").astype(str)
    english = sorted({m.group() for value in values for m in LATIN.finditer(value)})
    hebrew = sorted({
        value.strip() for value in values
        if HEBREW.search(value) and not LATIN.search(value) and value.strip()
    })
    if not english or not hebrew:
        continue

    # Compare English against Hebrew ONLY within the same field.
    vectors = model.encode(
        english + hebrew, batch_size=8, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )
    similarities = vectors[:len(english)] @ vectors[len(english):].T
    selected = {}

    for i, term in enumerate(english):
        top = int(np.argmax(similarities[i]))
        score = float(similarities[i, top])
        candidate = hebrew[top]
        if score > ALPHA:
            selected[term] = candidate

        scores.append({
            "Column": column, "English": term, "Best_Hebrew": candidate,
            "Cosine": round(score, 4), "Above_alpha": score > ALPHA,
        })

    # Inspect every populated cell, including mixed Hebrew/English text.
    for index, original in values.items():
        if not LATIN.search(original):
            continue
        proposed = LATIN.sub(
            lambda match: selected.get(match.group(), match.group()), original,
        )
        proposed_df.at[index, column] = proposed
        changed_cells.append({
            "ID": survey_df.at[index, "ID"], "Column": column,
            "Original": original, "Proposed": proposed,
            "Changed": proposed != original,
        })

# Review the highest candidate for each English expression and every affected cell.
scores_df = pd.DataFrame(scores).sort_values(
    ["Column", "Cosine"], ascending=[True, False],
)
changes_df = pd.DataFrame(changed_cells)
print("Highest Hebrew candidate per English expression (α =", ALPHA, ")")
display(scores_df)
print("All cells containing English; changes are proposals only:")
display(changes_df)
print("Proposed DataFrame:")
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(proposed_df)


In [ ]:
# Prepare the notebook's original and transformed tables for the analysis.
data = survey_df.replace("", pd.NA).copy()
transformed_data = proposed_df.replace("", pd.NA).copy()

# Restore numeric fields from the CSV strings using the metadata's integer types.
numeric_fields = ["ID", "MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha", "shnotlimud", "Gil"]
data[numeric_fields] = data[numeric_fields].apply(pd.to_numeric)
transformed_data[numeric_fields] = data[numeric_fields]

# Restore the workbook's zero title placeholder from its CSV representation.
data["TeurTafkid"] = data["TeurTafkid"].replace("0.0", "0")
transformed_data["TeurTafkid"] = transformed_data["TeurTafkid"].replace("0.0", "0")

# Continue with the identified survey records after the workplace transformation.
survey = transformed_data.loc[transformed_data["ID"].notna()].copy()

# Select the remaining fields in questionnaire order.
remaining_fields = ["SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid", "MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha", "shnotlimud", "Gil", "TarichHathala", "SemelAnafSofi", "SemelMishlachSofi"]

# Strip surrounding whitespace in the analysis values.
values = survey[remaining_fields].astype("string").apply(lambda column: column.str.strip().replace("", pd.NA))

# Count populated, empty, and distinct values for every remaining field.
field_summary = pd.DataFrame({"Populated": values.count(), "Empty": values.isna().sum(), "Distinct": values.nunique()})
display(field_summary)

# Identify the six remaining descriptive text fields.
text_fields = remaining_fields[:6]

# Identify full cells containing Latin letters.
latin_text = values[text_fields].apply(lambda column: column.str.contains(r"[A-Za-z]", na=False))

# Identify the reference expression present in this dataset.
references = values[text_fields].eq('כנ"ל').fillna(False)

# Count Latin-containing cells and reference answers by field.
display(pd.DataFrame({"Contains Latin letters": latin_text.sum(), "Reference answers": references.sum()}))

# Examine those answers alongside the other workplace and occupational descriptions.
display(survey.loc[latin_text.any(axis=1) | references.any(axis=1), ["ID", "ShemAvoda"] + text_fields])

# Identify workplace activities that exactly repeat the participant's work description.
repeated_activity = values["SugAvoda"].eq(values["EzoAvoda"]).fillna(False)

# Inspect whether the repeated answer describes the workplace or the participant.
display(survey.loc[repeated_activity, ["ID", "ShemAvoda", "SugAvoda", "EzoAvoda"]])

# Compare missing department names with missing department activities.
display(pd.crosstab(values["ShemMachlaka"].isna(), values["SugMachlaka"].isna(), rownames=["Department name empty"], colnames=["Department activity empty"]))

# Define the three fields asking directly about the participant's work.
role_fields = ["EzoAvoda", "TeurPeula", "TeurTafkid"]

# Examine other evidence when the main work description is absent.
display(survey.loc[values["EzoAvoda"].isna(), ["ID", "ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka"] + role_fields + ["SemelMishlachSofi"]])

# Identify records missing all three occupational descriptions.
no_role_text = values[role_fields].isna().all(axis=1)

# Display their occupation labels alongside the missing descriptions.
display(survey.loc[no_role_text, ["ID"] + role_fields + ["SemelMishlachSofi"]])

# Find job titles that repeat task descriptions.
repeated_title = values["TeurTafkid"].eq(values["TeurPeula"]).fillna(False)

# Examine those repetitions and the numeric title placeholder.
display(survey.loc[values["TeurTafkid"].eq("0").fillna(False) | repeated_title, ["ID"] + role_fields])

# Examine the observed disagreement between department manager and deputy department manager.
display(survey.loc[survey["ID"].eq(64), ["ID"] + role_fields + ["MenahelEtMi"]])

# Count management responses, including missing answers.
display(survey["MenahelEtMi"].value_counts(dropna=False).sort_index().rename("Records"))

# Inspect the managerial abbreviation recorded without a management response.
display(survey.loc[survey["ID"].eq(137), ["ID"] + role_fields + ["MenahelEtMi"]])

# Count the employment arrangements represented in the sample.
display(survey["MaamadAvoda"].value_counts(dropna=False).sort_index().rename("Records"))

# Retain the employment status and salary source for the eligibility check.
employment_status = survey["MaamadAvoda"]
salary_source = survey["MakorSachar"]

# Cross-tabulate employment status and salary source, including missing responses.
display(pd.crosstab(employment_status.astype("Int64").astype("string").fillna("Missing"), salary_source.astype("Int64").astype("string").fillna("Missing")))

# Identify salary-source answers outside the eligible employment codes 1 and 5.
salary_conflict = employment_status.notna() & ~employment_status.isin([1, 5]) & salary_source.notna()

# Display the contradictory combinations for examination.
display(survey.loc[salary_conflict, ["ID", "ShemAvoda", "MaamadAvoda", "MakorSachar"]])

# Compare qualification codes with reported years of schooling.
display(pd.crosstab(survey["TeudaGvoha"], survey["shnotlimud"]))

# Examine the observed seven-year schooling entries alongside their qualifications.
display(survey.loc[survey["shnotlimud"].eq(7), ["ID", "TeudaGvoha", "shnotlimud", "Gil"]])

# Summarize the two numerical measurements.
display(survey[["shnotlimud", "Gil"]].agg(["count", "min", "max", "median"]))

# Examine records missing education, age, or employment status.
display(survey.loc[survey[["TeudaGvoha", "shnotlimud", "Gil", "MaamadAvoda"]].isna().any(axis=1), ["ID", "TeudaGvoha", "shnotlimud", "Gil", "MaamadAvoda"]])

# Identify populated measurements that violate the declared integer format.
fractional_values = survey[["shnotlimud", "Gil"]].mod(1).ne(0) & survey[["shnotlimud", "Gil"]].notna()

# Examine fractional measurements and schooling durations exceeding recorded age.
display(survey.loc[fractional_values.any(axis=1) | survey["shnotlimud"].gt(survey["Gil"]), ["ID", "shnotlimud", "Gil"]])

# Parse the role-start dates for analysis.
start_dates = pd.to_datetime(survey["TarichHathala"], errors="coerce")

# Report their coverage, range, and number of distinct dates.
display(start_dates.agg(["count", "min", "max", "nunique"]))

# Display populated values that could not be parsed as dates.
display(survey.loc[survey["TarichHathala"].notna() & start_dates.isna(), ["ID", "TarichHathala"]])

# Remove a numeric storage suffix from the industry-code representation.
industry_codes = values["SemelAnafSofi"].str.replace(r"\.0$", "", regex=True)

# Restore the four positions specified by the metadata and Excel display format.
industry_codes = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4))

# Display the distribution of industry codes.
display(industry_codes.value_counts().rename("Records"))

# Examine partial industry codes alongside occupation labels.
display(survey.loc[industry_codes.str.contains("X", na=False), ["ID", "SemelAnafSofi", "SemelMishlachSofi"]])

# Remove a numeric storage suffix from the occupation-code representation.
occupation_codes = values["SemelMishlachSofi"].str.replace(r"\.0$", "", regex=True)

# Represent numeric occupation codes with four positions.
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))

# Identify complete four-digit occupation labels.
complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)

# Display the incomplete occupation-code patterns.
display(occupation_codes.loc[~complete_target].value_counts().rename("Records"))

# Count the records available for each complete occupation class.
class_counts = occupation_codes.loc[complete_target].value_counts()

# Quantify class support before choosing a validation scheme.
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}))

# Retrieve the original survey values for checking pre-existing label disagreements.
original_survey = data.loc[data["ID"].notna()].copy()

# Compare predictor values after trimming whitespace, excluding ID and the target.
predictor_values = original_survey.drop(columns=["ID", "SemelMishlachSofi"]).astype("string").apply(lambda column: column.str.strip().replace("", pd.NA))

# Group records with identical predictor profiles.
predictor_groups = predictor_values.fillna("<MISSING>").apply(tuple, axis=1)

# Identify identical profiles assigned different occupation labels.
conflicting_labels = occupation_codes.groupby(predictor_groups).transform("nunique").gt(1)

# Display the original records with conflicting labels.
display(original_survey.loc[conflicting_labels, ["ID", "ShemAvoda", "EzoAvoda", "SemelMishlachSofi"]])
